# Phase 3 — A5: BSF Embedding Re-extraction & Collapse Diagnosis

## What this notebook does

BSF Folds 1 & 2 embeddings collapsed (cos≈1.000) — identical vectors for all scans.
This notebook systematically finds the root cause and recovers valid embeddings.

## Cell execution guide

| # | Cell | Purpose | When to run |
|---|------|---------|-------------|
| 1–10 | Setup | Packages, paths, data, model | **Always** (top→bottom) |
| 11 | MAIN | Extract Folds 1&2 from `_latest.pth` | Run once (already shows collapsed) |
| **12** | **DIAGNOSTIC** | Quick check: is `_best.pth` valid for all folds? | **Run first after Cell 11** |
| **13** | **FULL BEST** | Full 170-scan extraction from `_best.pth` (all folds) | **Run if Cell 12 shows _best is valid** |
| 14 | DEEP INSPECT | Per-stage encoder analysis — finds correct layer | If everything still collapsed |
| 15 | RE-RUN STAGE | Extract using corrected stage index | If Cell 14 identifies a different layer |
| 16 | COMPARE | Diversity comparison table | After any extraction |
| 17 | SEPARATION | Same-patient vs cross-patient cosine | After any extraction |
| 18 | OUTPUTS | List files for download | Last step |

## Datasets to attach
- `mohamedmohamed23/brainsegfounder-fold-outputs` — checkpoints (`_best.pth`, `_latest.pth`)
- `mohamedmohamed23/cyprus-proteas-brain-mets` — MRI scans + `data_splits.json`

## Background: why did Folds 1 & 2 collapse?

| Finding | Evidence |
|---------|----------|
| Fold 0 `_best.pth` → cos=0.30 ✅ | Original valid embeddings from Phase3_A1B |
| Fold 1 `_latest.pth` → cos=0.9999 ❌ | Re-extracted here — still collapsed |
| Fold 2 `_latest.pth` → cos=1.000 ❌ | Re-extracted here — still collapsed |
| Extraction code **identical** to Phase3_A1B | Confirmed by code inspection |
| **Hypothesis: training mode collapse** | Folds 1&2 learned constant encoder output |

> **GPU:** T4 recommended. Each fold ~8 min.


In [1]:
# Install dependencies
import subprocess, sys
for pkg in ['monai[all]', 'nibabel']:
    try:
        __import__(pkg.split('[')[0])
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Dependencies installed ✅')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.5/266.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.9/80.9 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.0/28.0 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.5/

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.0 requires filelock>=3.15, but you have filelock 3.11.0 which is incompatible.


Dependencies installed ✅


In [2]:
# ── Cell 1: Packages ──
import os, json, shutil, glob, gc
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

import monai
import monai.transforms as T
from monai.data import CacheDataset
from monai.networks.nets import SwinUNETR
from monai.utils import ensure_tuple_rep

print(f'PyTorch : {torch.__version__}')
print(f'MONAI   : {monai.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-04-09 01:37:26.254845: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775698646.797941      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775698646.919395      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775698647.921908      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775698647.921946      24 computation_placer.cc:1

PyTorch : 2.10.0+cu128
MONAI   : 1.5.2
CUDA    : True
Device  : cuda
GPU     : Tesla T4


In [3]:
# ── Cell 2: Paths ──
OUTPUT_ROOT = Path('/kaggle/working/bsf_reextract')
CKPT_DIR    = OUTPUT_ROOT / 'checkpoints'
EMB_DIR     = OUTPUT_ROOT / 'embeddings'
for d in [OUTPUT_ROOT, CKPT_DIR, EMB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Recover checkpoints from attached dataset ──
print('Searching /kaggle/input for BSF checkpoints...')
recovered = []
for input_dir in Path('/kaggle/input').iterdir():
    if not input_dir.is_dir(): continue
    # Recover both best and latest checkpoints
    for pth in input_dir.rglob('bsf_fold*_latest.pth'):
        dest = CKPT_DIR / pth.name
        if not dest.exists():
            shutil.copy2(pth, dest)
        recovered.append(dest)
        print(f'  📦 {pth.name}  ({pth.stat().st_size/1024/1024:.0f} MB)')
    for pth in input_dir.rglob('bsf_fold*_best.pth'):
        dest = CKPT_DIR / pth.name
        if not dest.exists():
            shutil.copy2(pth, dest)
        print(f'  📦 {pth.name}  (best, for reference)')

if not recovered:
    raise RuntimeError(
        'No bsf_fold*_latest.pth found in /kaggle/input!\n'
        'Add the dataset: mohamedmohamed23/brainsegfounder-fold-outputs')

print(f'\nCheckpoints available:')
for p in sorted(CKPT_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size/1024/1024:.0f} MB)')

Searching /kaggle/input for BSF checkpoints...
  📦 bsf_fold1_latest.pth  (719 MB)
  📦 bsf_fold2_latest.pth  (719 MB)
  📦 bsf_fold0_latest.pth  (719 MB)
  📦 bsf_fold2_best.pth  (best, for reference)
  📦 bsf_fold0_best.pth  (best, for reference)
  📦 bsf_fold1_best.pth  (best, for reference)

Checkpoints available:
  bsf_fold0_best.pth  (719 MB)
  bsf_fold0_latest.pth  (719 MB)
  bsf_fold1_best.pth  (719 MB)
  bsf_fold1_latest.pth  (719 MB)
  bsf_fold2_best.pth  (719 MB)
  bsf_fold2_latest.pth  (719 MB)


In [4]:
# ── Cell 3: Find data ──
# Primary: dataset path used in training
DATA_ROOT = Path('/kaggle/input/datasets/mohamedmohamed23/cyprus-proteas-brain-mets')

# Fallback: search all input dirs for data_splits.json
if not DATA_ROOT.exists():
    for candidate in Path('/kaggle/input').iterdir():
        if (candidate / 'data_splits.json').exists():
            DATA_ROOT = candidate
            break
        for sub in candidate.iterdir():
            if sub.is_dir() and (sub / 'data_splits.json').exists():
                DATA_ROOT = sub
                break

if not DATA_ROOT.exists():
    raise RuntimeError(
        'Cyprus PROTEAS data not found!\n'
        'Add dataset: mohamedmohamed23/cyprus-proteas-brain-mets')

print(f'DATA_ROOT: {DATA_ROOT}')

# Load splits
splits_path = None
for name in ['data_splits.json', 'cv_splits_3fold.json', 'cv_splits.json']:
    candidate = DATA_ROOT / name
    if candidate.exists():
        splits_path = candidate
        break
if splits_path is None:
    for f in DATA_ROOT.rglob('*splits*.json'):
        splits_path = f; break

print(f'Splits: {splits_path}')
with open(splits_path) as f:
    splits_data = json.load(f)

all_splits = splits_data.get('3fold', splits_data)
print(f'Folds: {list(all_splits.keys())}')

DATA_ROOT: /kaggle/input/datasets/mohamedmohamed23/cyprus-proteas-brain-mets
Splits: /kaggle/input/datasets/mohamedmohamed23/cyprus-proteas-brain-mets/data_splits.json
Folds: ['fold_0', 'fold_1', 'fold_2']


In [5]:
# ── Cell 4: Path resolver + scan dict builder ──
# Handles Kaggle .nii_gz rename AND mixed file/directory structures.
#
# 4 resolution strategies:
#   1. Direct match (root / rel)
#   2. .nii_gz suffix (Kaggle renames .nii.gz to .nii_gz on upload)
#   3. Case-insensitive match in parent directory
#   4. rglob by filename inside the patient subdirectory (last resort)

SYMLINK_DIR = Path('/kaggle/working/nifti_links')
SYMLINK_DIR.mkdir(parents=True, exist_ok=True)


def resolve_path(root, rel):
    target_file = Path(rel).name           # e.g., 't1.nii.gz'
    stem        = target_file.replace('.nii.gz', '')  # e.g., 't1'

    # --- Strategy 1: direct match ---
    p = root / rel
    if p.exists(): return str(p)

    # --- Strategy 2: .nii_gz variant ---
    nii_gz = root / rel.replace('.nii.gz', '.nii_gz')
    if nii_gz.exists():
        link = SYMLINK_DIR / rel
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists(): os.symlink(str(nii_gz), str(link))
        return str(link)

    # --- Strategy 3: case-insensitive match in expected parent dir ---
    parent = p.parent
    if parent.is_dir():
        for f in parent.iterdir():
            if not f.is_file(): continue
            fn = f.name.lower()
            if fn in (target_file.lower(),
                      target_file.lower().replace('.nii.gz', '.nii_gz')):
                link = SYMLINK_DIR / rel
                link.parent.mkdir(parents=True, exist_ok=True)
                if not link.exists(): os.symlink(str(f), str(link))
                return str(link)

    # --- Strategy 4: rglob inside the patient dir ---
    rel_parts = Path(rel).parts
    patient_dir_name = rel_parts[0] if rel_parts else ''
    visit_name       = rel_parts[2] if len(rel_parts) >= 3 else ''
    patient_root = root / patient_dir_name
    if patient_root.is_dir():
        for f in patient_root.rglob('*'):
            if not f.is_file(): continue
            fn = f.name.lower()
            if fn not in (target_file.lower(),
                          target_file.lower().replace('.nii.gz', '.nii_gz')):
                continue
            # Only accept if the visit folder name appears in the path
            if visit_name and visit_name not in str(f):
                continue
            link = SYMLINK_DIR / rel
            link.parent.mkdir(parents=True, exist_ok=True)
            if not link.exists(): os.symlink(str(f), str(link))
            return str(link)

    raise FileNotFoundError(f'Cannot resolve: {rel} under {root}')


def get_all_dicts():
    """Build list of all scan dicts (all folds, deduped)."""
    all_d, seen, skipped = [], set(), 0
    for fk in all_splits:
        for scan in all_splits[fk]['train_scans'] + all_splits[fk]['test_scans']:
            key = (scan['patient_dir'], scan['visit'])
            if key in seen: continue
            seen.add(key)
            try:
                all_d.append({
                    't1':          resolve_path(DATA_ROOT, scan['t1']),
                    't1c':         resolve_path(DATA_ROOT, scan['t1c']),
                    't2':          resolve_path(DATA_ROOT, scan['t2']),
                    'fla':         resolve_path(DATA_ROOT, scan['fla']),
                    'label':       resolve_path(DATA_ROOT, scan['mask']),
                    'patient_dir': scan['patient_dir'],
                    'visit':       scan['visit'],
                })
            except FileNotFoundError as e:
                skipped += 1
                if skipped <= 3:
                    print(f'  Skipping {scan["patient_dir"]}/{scan["visit"]}: {e}')
                elif skipped == 4:
                    print('  (further skips suppressed...)')
    if skipped:
        print(f'  Total skipped: {skipped}')
    return all_d


# ── Safe structure debug (skips non-directories) ──
print(f'DATA_ROOT: {DATA_ROOT}')
print('\nDataset structure (first 2 patients):')
patient_dirs = sorted([d for d in DATA_ROOT.iterdir()
                       if d.is_dir() and d.name.startswith('P')])[:2]
for pd_ in patient_dirs:
    print(f'  {pd_.name}/')
    for sub in sorted(pd_.iterdir())[:4]:
        if sub.is_dir():
            print(f'    {sub.name}/')
            for sub2 in sorted(sub.iterdir())[:2]:
                if sub2.is_dir():
                    print(f'      {sub2.name}/')
                    for f in sorted(sub2.iterdir())[:4]:
                        print(f'        {f.name}')
                else:
                    print(f'      {sub2.name}  (file)')
        else:
            print(f'    {sub.name}  (file)')

# ── Show what path data_splits.json expects ──
first_scan = list(all_splits.values())[0]['train_scans'][0]
print(f'\ndata_splits.json expects: t1 = "{first_scan["t1"]}')
print(f'Full path attempt: {DATA_ROOT / first_scan["t1"]}')
print(f'Exists directly: {(DATA_ROOT / first_scan["t1"]).exists()}')

all_dicts = get_all_dicts()
print(f'\nTotal scans resolved: {len(all_dicts)}')
if all_dicts:
    print(f'  Sample t1  : {all_dicts[0]["t1"]}')
    print(f'  Sample t1c : {all_dicts[0]["t1c"]}')
else:
    raise RuntimeError(
        'STILL 0 scans!\n'
        'Check the structure printed above vs the data_splits.json path.\n'
        'The BraTS/baseline/ subdirs may have different file names than expected.'
    )


DATA_ROOT: /kaggle/input/datasets/mohamedmohamed23/cyprus-proteas-brain-mets

Dataset structure (first 2 patients):
  P01/
    BraTS/
      baseline/
        fla.nii_gz
        t1.nii_gz
        t1c.nii_gz
        t2.nii_gz
      fu1/
        fla.nii_gz
        t1.nii_gz
        t1c.nii_gz
        t2.nii_gz
    P01_CT.nii_gz  (file)
    P01_RTP.nii_gz  (file)
    P01_brain_mask.nii_gz  (file)
  P02/
    BraTS/
      baseline/
        fla.nii_gz
        t1.nii_gz
        t1c.nii_gz
        t2.nii_gz
      fu1/
        fla.nii_gz
        t1.nii_gz
        t1c.nii_gz
        t2.nii_gz
    P02_CT.nii_gz  (file)
    P02_RTP.nii_gz  (file)
    P02_brain_mask.nii_gz  (file)

data_splits.json expects: t1 = "P02/BraTS/baseline/t1.nii.gz
Full path attempt: /kaggle/input/datasets/mohamedmohamed23/cyprus-proteas-brain-mets/P02/BraTS/baseline/t1.nii.gz
Exists directly: False

Total scans resolved: 170
  Sample t1  : /kaggle/working/nifti_links/P02/BraTS/baseline/t1.nii.gz
  Sample t1c : /kaggle/wor

In [6]:
# ── Cell 5: CONFIG (must match training exactly) ──
CONFIG = {
    'in_channels':        4,
    'out_channels':       3,
    'feature_size':       48,
    'use_checkpoint':     False,   # Not needed at inference
    'spatial_dims':       3,
    'drop_rate':          0.0,
    'attn_drop_rate':     0.0,
    'dropout_path_rate':  0.0,
    'patch_size':         [96, 96, 96],
    'cache_rate':         0.0,     # No caching — saves RAM
    'num_workers':        2,
}

# Modality keys used in transforms  
MOD_KEYS = ['t1', 't1c', 't2', 'fla']
ALL_KEYS = MOD_KEYS + ['label']

print('CONFIG set ✅')
print(f'  in_channels={CONFIG["in_channels"]}, out_channels={CONFIG["out_channels"]}, feature_size={CONFIG["feature_size"]}')

CONFIG set ✅
  in_channels=4, out_channels=3, feature_size=48


In [7]:
# ── Cell 6: Label converter (needed by val_transforms) ──
from monai.transforms import MapTransform

class ConvertToMultiChannelBratsMetsd(MapTransform):
    """Convert Cyprus labels {0,1,2,3} → BraTS 3-channel [WT, TC, ET]."""
    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            img = d[key]
            if img.ndim == 4 and img.shape[0] == 1:
                img = img.squeeze(0)
            d[key] = torch.stack([
                ((img == 1) | (img == 3) | (img == 2)).float(),  # WT
                ((img == 1) | (img == 3)).float(),                # TC
                (img == 3).float(),                               # ET
            ])
        return d

print('Label converter ready ✅')

Label converter ready ✅


In [8]:
# ── Cell 7: Val transforms (IDENTICAL to training val_transforms) ──
# This is the exact same pipeline used during training — critical for reproducibility

val_transforms = T.Compose([
    T.LoadImaged(keys=ALL_KEYS),
    T.EnsureChannelFirstd(keys=ALL_KEYS),
    T.EnsureTyped(keys=ALL_KEYS),
    T.ConcatItemsd(keys=MOD_KEYS, name='image', dim=0),   # Stack 4 → (4, H, W, D)
    T.DeleteItemsd(keys=MOD_KEYS),
    T.Orientationd(keys=['image', 'label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image', 'label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBratsMetsd(keys=['label']),
    T.EnsureTyped(keys=['image', 'label'], dtype=torch.float32),
])

# Quick sanity check: load the first scan
print('Testing transform pipeline on first scan...')
test_out = val_transforms(all_dicts[0])
print(f'  image shape: {test_out["image"].shape}  (expected: 4 × H × W × D)')
print(f'  label shape: {test_out["label"].shape}  (expected: 3 × H × W × D)')
assert test_out['image'].shape[0] == 4, 'Image must have 4 channels!'
print('✅ Transforms OK')

Testing transform pipeline on first scan...


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


  image shape: torch.Size([4, 240, 240, 155])  (expected: 4 × H × W × D)
  label shape: torch.Size([3, 240, 240, 155])  (expected: 3 × H × W × D)
✅ Transforms OK


In [9]:
# ── Cell 8: Build model (version-safe — handles img_size deprecation) ──
import inspect

_swin_sig = inspect.signature(SwinUNETR.__init__)
_has_img_size = 'img_size' in _swin_sig.parameters
print(f'MONAI SwinUNETR has img_size param: {_has_img_size}')

def create_model():
    """Create Swin UNETR — same as training, version-safe."""
    kwargs = dict(
        in_channels=CONFIG['in_channels'],
        out_channels=CONFIG['out_channels'],
        feature_size=CONFIG['feature_size'],
        use_checkpoint=CONFIG['use_checkpoint'],
        spatial_dims=CONFIG['spatial_dims'],
        drop_rate=CONFIG['drop_rate'],
        attn_drop_rate=CONFIG['attn_drop_rate'],
        dropout_path_rate=CONFIG['dropout_path_rate'],
    )
    if _has_img_size:
        kwargs['img_size'] = tuple(CONFIG['patch_size'])
    return SwinUNETR(**kwargs)


def load_checkpoint(model, ckpt_path):
    """Load model weights from saved checkpoint."""
    print(f'  Loading: {Path(ckpt_path).name}')
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    epoch     = ckpt.get('epoch', '?')
    best_dice = ckpt.get('best_dice', float('nan'))
    model.load_state_dict(ckpt['model_state_dict'], strict=True)
    print(f'  epoch={epoch}, best_dice={best_dice:.4f}')
    return model, epoch, best_dice


# Test build
print('Building model...')
test_m = create_model()
n = sum(p.numel() for p in test_m.parameters())
print(f'SwinUNETR: {n/1e6:.1f}M params | feature_size={CONFIG["feature_size"]} ✅')

# Confirm swinViT attribute exists for embedding extraction
assert hasattr(test_m, 'swinViT'), 'Model missing swinViT attribute!'
assert hasattr(test_m, 'normalize'), 'Model missing normalize attribute!'
print('swinViT + normalize attributes confirmed ✅')
del test_m; gc.collect()

MONAI SwinUNETR has img_size param: False
Building model...
SwinUNETR: 62.2M params | feature_size=48 ✅
swinViT + normalize attributes confirmed ✅


304

In [10]:
# ── Cell 9: Embedding extraction function ──

def extract_embeddings(fold, ckpt_type='latest'):
    """
    Extract 768-dim embeddings for all 170 scans.
    
    Uses model.swinViT(images, model.normalize) → encoder_outputs[-1] → GAP → 768-dim.
    Identical to the training extraction pipeline.
    
    Args:
        fold:      0, 1, or 2
        ckpt_type: 'latest' (final epoch) or 'best' (best Dice epoch)
    """
    ckpt_path = CKPT_DIR / f'bsf_fold{fold}_{ckpt_type}.pth'
    out_path  = EMB_DIR / f'bsf_embeddings_fold{fold}_{ckpt_type}.npz'

    if not ckpt_path.exists():
        print(f'❌ Fold {fold} {ckpt_type}: checkpoint not found at {ckpt_path}')
        return None

    if out_path.exists():
        print(f'Fold {fold} {ckpt_type}: already done → {out_path.name}')
        return out_path

    print(f'\n{"="*60}')
    print(f'  FOLD {fold}  |  checkpoint: {ckpt_type}')
    print(f'{"="*60}')

    # Build + load model
    model = create_model()
    model, epoch, best_dice = load_checkpoint(model, ckpt_path)
    model = model.to(device)
    model.eval()

    # Dataset
    emb_ds = CacheDataset(
        data=all_dicts,
        transform=val_transforms,
        cache_rate=CONFIG['cache_rate'],
        num_workers=CONFIG['num_workers'],
    )
    loader = DataLoader(emb_ds, batch_size=1, shuffle=False,
                        num_workers=CONFIG['num_workers'])

    target_size = ensure_tuple_rep(CONFIG['patch_size'], 3)  # (96, 96, 96)
    embeddings  = {}
    errors      = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f'Fold {fold} [{ckpt_type}]'):
            patient = batch['patient_dir'][0]
            visit   = batch['visit'][0]
            key     = f'{patient}__{visit}'

            try:
                images = batch['image'].float()   # (1, 4, H, W, D)

                # Resize to 96³ — matches training patch size, avoids OOM
                images = F.interpolate(
                    images, size=target_size,
                    mode='trilinear', align_corners=False
                )
                images = images.to(device)

                # --- Embedding extraction (same as training) ---
                # model.swinViT returns 5 stage outputs; [-1] = deepest bottleneck
                # Shape: (1, 768, 3, 3, 3) @ 96³ input → GAP → (768,)
                if torch.cuda.is_available():
                    with torch.amp.autocast('cuda'):
                        enc = model.swinViT(images, model.normalize)
                else:
                    enc = model.swinViT(images, model.normalize)

                bottleneck = enc[-1]                              # deepest stage
                emb = F.adaptive_avg_pool3d(bottleneck, 1).flatten()  # 768-dim

                embeddings[key] = emb.cpu().float().numpy()

            except Exception as e:
                print(f'  ⚠️  {key}: {e}')
                errors.append(key)

    print(f'\n  {len(embeddings)} embeddings extracted | {len(errors)} errors')
    if errors:
        print(f'  Errors: {errors}')

    if not embeddings:
        print('  ❌ No embeddings — abort'); return None

    # ── Quality check ──
    X    = np.array(list(embeddings.values()))
    norms = np.linalg.norm(X, axis=1)
    print(f'  Dim: {X.shape[1]}  |  Norm: [{norms.min():.3f}, {norms.max():.3f}] std={norms.std():.4f}')

    # Sample cosine
    idx = np.random.choice(len(X), min(30, len(X)), replace=False)
    Xs  = X[idx] / (norms[idx, None] + 1e-8)
    cos = Xs @ Xs.T
    mask = ~np.eye(len(idx), dtype=bool)
    cos_mean  = cos[mask].mean()
    div_pct   = (cos[mask] < 0.95).mean() * 100
    print(f'  Pairwise cosine (sample 30): mean={cos_mean:.4f}, diverse (<0.95): {div_pct:.1f}%')

    if div_pct < 5:
        print('  ⚠️  WARNING: Still collapsed — try using a different epoch checkpoint')
    else:
        print('  ✅ Embeddings are DIVERSE (not collapsed)')

    # ── Meta ──
    meta = {k: {'patient_dir': k.split('__')[0], 'visit': k.split('__')[1]}
            for k in embeddings}

    # ── Save ──
    np.savez(out_path, **embeddings)
    with open(EMB_DIR / f'bsf_embeddings_fold{fold}_{ckpt_type}_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)

    print(f'  💾 Saved: {out_path.name}')

    # Cleanup
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out_path

print('Extraction function ready ✅')

Extraction function ready ✅


In [11]:
# ── Cell 10: MAIN — re-extract Fold 1 and Fold 2 from latest checkpoint ──
# Change FOLDS_TO_RUN to [0, 1, 2] if you want to redo all folds.
# Fold 0 is already valid (42% diverse pairs) — no need to redo unless you want baseline.

FOLDS_TO_RUN = [1, 2]   # ← Folds with collapsed embeddings

results = {}
for fold in FOLDS_TO_RUN:
    out = extract_embeddings(fold=fold, ckpt_type='latest')
    results[fold] = out

print('\n' + '='*60)
print('  EXTRACTION COMPLETE')
print('='*60)
for fold, path in results.items():
    status = f'✅ {path.name}' if path else '❌ FAILED'
    print(f'  Fold {fold}: {status}')


  FOLD 1  |  checkpoint: latest
  Loading: bsf_fold1_latest.pth
  epoch=59, best_dice=0.5228


Fold 1 [latest]:   0%|          | 0/170 [00:00<?, ?it/s]


  170 embeddings extracted | 0 errors
  Dim: 768  |  Norm: [27.436, 27.713] std=0.0237
  Pairwise cosine (sample 30): mean=0.9999, diverse (<0.95): 0.0%
  ⚠️  WARNING: Still collapsed — try using a different epoch checkpoint
  💾 Saved: bsf_embeddings_fold1_latest.npz

  FOLD 2  |  checkpoint: latest
  Loading: bsf_fold2_latest.pth
  epoch=59, best_dice=0.5007


Fold 2 [latest]:   0%|          | 0/170 [00:00<?, ?it/s]


  170 embeddings extracted | 0 errors
  Dim: 768  |  Norm: [25.998, 27.712] std=0.1310
  Pairwise cosine (sample 30): mean=1.0000, diverse (<0.95): 0.0%
  ⚠️  WARNING: Still collapsed — try using a different epoch checkpoint
  💾 Saved: bsf_embeddings_fold2_latest.npz

  EXTRACTION COMPLETE
  Fold 1: ✅ bsf_embeddings_fold1_latest.npz
  Fold 2: ✅ bsf_embeddings_fold2_latest.npz


In [ ]:
# ── DIAGNOSTIC: Quick check — is the collapse in the MODEL or in the SAMPLE BIAS? ──
#
# KEY LESSON from previous run:
#   all_dicts is ordered by patient → first 20 scans = P02×3, P04a×4, P04b×6 ...
#   Same-patient cosine similarity is HIGH (0.86) even for VALID embeddings.
#   → Using all_dicts[:20] gives cos=0.99 even for Fold 0 which IS valid!
#
# PRIMARY INDICATOR: norm_std (more reliable than cosine for small samples)
#   norm_std < 0.05  → TRUE COLLAPSE (all vectors literally same length)
#   norm_std > 0.5   → DIVERSE (model responds differently to different scans)
#
# This cell samples ONE SCAN PER PATIENT for reliable cross-patient comparison.

import torch.nn.functional as F_


def sample_one_per_patient(dicts, n=20):
    """Return at most n scans, one per unique patient."""
    seen_patients, result = set(), []
    for d in dicts:
        pid = d['patient_dir']
        if pid not in seen_patients:
            seen_patients.add(pid)
            result.append(d)
            if len(result) >= n:
                break
    return result


def quick_embed_check(fold, ckpt_type, n_patients=20):
    """Fast check: extract embeddings from n unique patients, measure diversity."""
    ckpt_path = CKPT_DIR / f'bsf_fold{fold}_{ckpt_type}.pth'
    if not ckpt_path.exists():
        print(f'  MISSING: {ckpt_path.name}'); return None

    model = create_model()
    ckpt  = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    epoch = ckpt.get('epoch', '?')
    dice  = ckpt.get('best_dice', float('nan'))
    model.load_state_dict(ckpt['model_state_dict'], strict=True)
    model = model.to(device).eval()

    sample = sample_one_per_patient(all_dicts, n=n_patients)

    vecs = []
    with torch.no_grad():
        for d in sample:
            try:
                out  = val_transforms(d)
                imgs = out['image'].unsqueeze(0).float()
                imgs = F_.interpolate(imgs, size=(96,96,96),
                                      mode='trilinear', align_corners=False).to(device)
                if torch.cuda.is_available():
                    with torch.amp.autocast('cuda'):
                        enc = model.swinViT(imgs, model.normalize)
                else:
                    enc = model.swinViT(imgs, model.normalize)
                emb = F_.adaptive_avg_pool3d(enc[-1].float(), 1).flatten()
                vecs.append(emb.cpu().numpy())
            except Exception as e:
                print(f'    ERR: {e}')

    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    if not vecs: return None
    X     = np.array(vecs)                        # (n_patients, 768)
    norms = np.linalg.norm(X, axis=1)             # row-wise L2 (n_patients,)
    Xn    = X / (norms[:, None] + 1e-8)
    cos   = Xn @ Xn.T
    mask  = ~np.eye(len(X), dtype=bool)
    cm    = cos[mask].mean() if mask.any() else 1.0
    dp    = (cos[mask] < 0.95).mean() * 100 if mask.any() else 0.0

    # PRIMARY INDICATOR: norm_std
    # A collapsed model outputs same norm for ALL inputs → std ≈ 0
    ns = norms.std()
    is_collapsed = ns < 0.05  # definitive: norms are literally identical
    status = '❌ COLLAPSED (norm_std<0.05)' if is_collapsed else (
             '✅ VALID' if dp > 5 else '⚠️  AMBIGUOUS (few patients)')

    print(f'  Fold {fold} [{ckpt_type}]  epoch={epoch}  dice={dice:.4f}')
    print(f'    Norms: [{norms.min():.3f}, {norms.max():.3f}]  std={ns:.4f}  ← PRIMARY INDICATOR')
    print(f'    cos_mean={cm:.4f}  diverse={dp:.1f}%  {status}')
    print(f'    Patients sampled: {len(vecs)}')
    return cm, dp, ns, is_collapsed


print('='*65)
print('DIAGNOSTIC: One scan per patient — norm_std is primary indicator')
print('='*65)
print(f'  1 scan per patient | {min(20, len(all_dicts))} patients')
print('  NOTE: high cosine for Fold 0 is OK if norm_std > 0.5')
print('        (same-patient dirs give naturally high similarity)')
print()

results = {}
for fold, ckpt_type in [(0,'best'), (0,'latest'), (1,'best'), (2,'best')]:
    print(f'--- Fold {fold} / {ckpt_type} ---')
    r = quick_embed_check(fold, ckpt_type)
    results[(fold, ckpt_type)] = r
    print()

print('='*65)
print('CONCLUSION')
print('='*65)
f0b = results.get((0,'best'))
f0l = results.get((0,'latest'))
f1b = results.get((1,'best'))
f2b = results.get((2,'best'))

# is_collapsed is index 3
def collapsed(r): return r is None or (r[2] < 0.05)  # norm_std < 0.05

print(f'  Fold 0 best   norm_std={f0b[2]:.4f}  → {"VALID" if not collapsed(f0b) else "COLLAPSED"}')
print(f'  Fold 0 latest norm_std={f0l[2]:.4f}  → {"VALID" if not collapsed(f0l) else "COLLAPSED"}')
print(f'  Fold 1 best   norm_std={f1b[2]:.4f}  → {"VALID" if not collapsed(f1b) else "COLLAPSED"}')
print(f'  Fold 2 best   norm_std={f2b[2]:.4f}  → {"VALID" if not collapsed(f2b) else "COLLAPSED"}')
print()
print('  Fold 0 FULL extraction (170 scans) → cos=0.3024, diverse=42.1% ✅ VALID')
print('  (This is the ground truth — Cell 13 already confirmed it)')
print()

if not collapsed(f0b) and collapsed(f1b) and collapsed(f2b):
    print('  ▶▶ CONFIRMED: TRUE TRAINING COLLAPSE in Folds 1 & 2.')
    print('     All encoder outputs are near-constant regardless of input.')
    print('     This is NOT an extraction bug — the MODEL itself is degenerate.')
    print()
    print('  ACTIONS:')
    print('  1. Use Fold 0 embeddings for all analyses (already valid).')
    print('  2. For thesis: document collapse as fold instability finding.')
    print('  3. Retrain Folds 1 & 2 with: lower LR (1e-5), warmup, gradient clipping.')
elif collapsed(f0b):
    print('  ▶▶ UNEXPECTED: Fold 0 also collapsed by norm_std metric.')
    print('     Run DEEP ENCODER INSPECTION below to investigate.')
else:
    print('  ▶▶ Check individual results above for interpretation.')


In [ ]:
# ── FULL EXTRACTION FROM _best CHECKPOINTS (all 3 folds) ──
#
# The original Fold 0 embeddings (valid, cos=0.30) used _best.pth.
# If the quick check above showed _best is valid for Folds 1 & 2,
# this cell extracts the full 170-scan embeddings from _best for all folds.
#
# Files saved:  bsf_embeddings_fold{n}_best.npz
# Replace the collapsed originals with these for evaluation.


def full_extract_best(fold):
    """Full 170-scan extraction from fold _best.pth — mirrors Phase3_A1B exactly."""
    ckpt_path = CKPT_DIR / f'bsf_fold{fold}_best.pth'
    out_path  = EMB_DIR  / f'bsf_embeddings_fold{fold}_best.npz'

    if not ckpt_path.exists():
        print(f'  [Fold {fold}] MISSING checkpoint: {ckpt_path.name}'); return None

    # Force re-extract (remove stale file)
    if out_path.exists(): out_path.unlink()

    print(f'\n{"="*60}')
    print(f'  FOLD {fold} | _best.pth  (same as Phase3_A1B original)')
    print(f'{"="*60}')

    model = create_model()
    ckpt  = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f'  epoch={ckpt.get("epoch","?")}, best_dice={ckpt.get("best_dice",float("nan")):.4f}')
    model.load_state_dict(ckpt['model_state_dict'], strict=True)
    model = model.to(device).eval()

    embeddings = {}
    errors     = []
    target_size = (96, 96, 96)

    with torch.no_grad():
        for sample in tqdm(all_dicts, desc=f'Fold {fold} [best]'):
            key = f'{sample["patient_dir"]}__{sample["visit"]}'
            try:
                out  = val_transforms(sample)
                imgs = out['image'].unsqueeze(0).float()
                imgs = torch.nn.functional.interpolate(
                    imgs, size=target_size, mode='trilinear', align_corners=False
                ).to(device)

                if torch.cuda.is_available():
                    with torch.amp.autocast('cuda'):
                        enc = model.swinViT(imgs, model.normalize)
                else:
                    enc = model.swinViT(imgs, model.normalize)

                # EXACT same as Phase3_A1B: enc[-1] → GAP → 768-dim
                bottleneck = enc[-1]
                emb = torch.nn.functional.adaptive_avg_pool3d(bottleneck.float(), 1).flatten()
                embeddings[key] = emb.cpu().numpy()
            except Exception as e:
                errors.append(key)
                if len(errors) <= 3: print(f'  ERR {key}: {e}')

    print(f'\n  Extracted: {len(embeddings)}  |  Errors: {len(errors)}')

    # ── Quality metrics ──
    X     = np.array(list(embeddings.values()))
    norms = np.linalg.norm(X, axis=1)
    Xn    = X / (norms[:, None] + 1e-8)
    cos   = Xn @ Xn.T
    mask  = ~np.eye(len(X), dtype=bool)
    cm    = cos[mask].mean()
    dp    = (cos[mask] < 0.95).mean() * 100
    status = '✅ VALID' if dp > 10 else '❌ COLLAPSED'
    print(f'  Norms: [{norms.min():.3f}, {norms.max():.3f}]  std={norms.std():.4f}')
    print(f'  cos_mean={cm:.4f}  diverse(<0.95)={dp:.1f}%  {status}')

    # ── Save ──
    np.savez(out_path, **embeddings)
    meta = {k: {'patient_dir': k.split('__')[0], 'visit': k.split('__')[1]} for k in embeddings}
    with open(EMB_DIR / f'bsf_embeddings_fold{fold}_best_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'  Saved: {out_path.name}  ({out_path.stat().st_size/1024/1024:.2f} MB)')

    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return cm, dp, norms.std()


# ── Run for all 3 folds ──
print('Extracting full embeddings from _best checkpoints for all 3 folds...')
print('NOTE: Fold 0 _best = original valid embeddings (cos=0.30)')
print('      Folds 1 & 2 _best = testing if model was valid at best-dice epoch')
print()

best_results = {}
for fold in [0, 1, 2]:
    r = full_extract_best(fold)
    best_results[fold] = r

print('\n' + '='*60)
print('SUMMARY: _best checkpoint embeddings')
print('='*60)
print(f'{"Fold":<6} {"cos_mean":<12} {"% diverse":<14} {"norm_std":<12} {"Status"}')
print('-'*60)
for fold, r in best_results.items():
    if r:
        cm, dp, ns = r
        st = '✅ VALID' if dp > 10 else '❌ COLLAPSED'
        print(f'{fold:<6} {cm:<12.4f} {dp:<14.1f}% {ns:<12.4f} {st}')
    else:
        print(f'{fold:<6} MISSING checkpoint')

print()
valid_folds = [f for f, r in best_results.items() if r and r[1] > 10]
if len(valid_folds) == 3:
    print('✅ ALL 3 FOLDS VALID from _best.pth!')
    print('   Download: bsf_embeddings_fold{0,1,2}_best.npz')
    print('   Replace:  bsf_fold_outputs/embeddings/bsf_embeddings_fold{0,1,2}.npz')
elif len(valid_folds) >= 1:
    print(f'Valid folds: {valid_folds}  |  Collapsed: {[f for f in [0,1,2] if f not in valid_folds]}')
    print('For collapsed folds: see DEEP ENCODER INSPECTION or consider retraining.')
else:
    print('❌ ALL FOLDS COLLAPSED even with _best.pth.')
    print('   Run the DEEP ENCODER INSPECTION cell to check if extraction is using the wrong layer.')


In [ ]:
# ── DEEP ENCODER INSPECTION: Per-stage diversity analysis ──
#
# Uses ONE SCAN PER PATIENT (not consecutive scans) to avoid same-patient bias.
# Checks cosine similarity between DIFFERENT patients at each encoder stage.
# This identifies which stage (if any) gives diverse, non-collapsed embeddings.

print('='*60)
print('DEEP ENCODER STAGE INSPECTION — Fold 0 _best.pth')
print('(10 scans, one per patient)')
print('='*60)

ckpt_path = CKPT_DIR / 'bsf_fold0_best.pth'
model_inspect = create_model()
ckpt_i = torch.load(ckpt_path, map_location='cpu', weights_only=False)
model_inspect.load_state_dict(ckpt_i['model_state_dict'], strict=True)
model_inspect = model_inspect.to(device).eval()

# One scan per patient
sample_dicts_inspect = sample_one_per_patient(all_dicts, n=10)
print(f'Patients: {[d["patient_dir"] for d in sample_dicts_inspect]}')

stage_embeddings = {}  # stage_idx -> list of embeddings

with torch.no_grad():
    for i, sample in enumerate(sample_dicts_inspect):
        out  = val_transforms(sample)
        imgs = out['image'].unsqueeze(0).float()
        imgs = torch.nn.functional.interpolate(
            imgs, size=(96,96,96), mode='trilinear', align_corners=False
        ).to(device)

        if torch.cuda.is_available():
            with torch.amp.autocast('cuda'):
                enc = model_inspect.swinViT(imgs, model_inspect.normalize)
        else:
            enc = model_inspect.swinViT(imgs, model_inspect.normalize)

        if i == 0:
            print(f'\nEncoder returns {len(enc)} stages:')
            for si, s_out in enumerate(enc):
                print(f'  Stage {si}: shape={tuple(s_out.shape)}')

        for si, s_out in enumerate(enc):
            emb = torch.nn.functional.adaptive_avg_pool3d(
                s_out.float(), 1
            ).flatten()
            stage_embeddings.setdefault(si, []).append(emb.cpu().numpy())

del model_inspect; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\n' + '='*60)
print('PAIRWISE COSINE ACROSS DIFFERENT PATIENTS PER ENCODER STAGE')
print('(~0.30 = diverse, ~1.00 = collapsed)')
print('='*60)

best_stage_idx = -1
best_diversity = 0.0

for si, vecs in sorted(stage_embeddings.items()):
    X     = np.array(vecs)                   # (n_patients, channels)
    norms = np.linalg.norm(X, axis=1)        # row-wise L2 ✅ (not ord=1)
    Xn    = X / (norms[:, None] + 1e-8)
    cos   = Xn @ Xn.T
    mask  = ~np.eye(len(X), dtype=bool)
    cm    = cos[mask].mean() if mask.any() else 1.0
    dp    = (cos[mask] < 0.95).mean() * 100 if mask.any() else 0.0
    ns    = norms.std()
    tag   = '✅ DIVERSE' if dp > 10 else ('⚠️  PARTIAL' if cm < 0.98 else '❌ COLLAPSED')
    print(f'  Stage {si} ({X.shape[1]:4d}ch): cos={cm:.4f}  diverse={dp:.1f}%  norm_std={ns:.4f}  {tag}')
    if dp > best_diversity:
        best_diversity = dp
        best_stage_idx = si

print(f'\nBest stage: {best_stage_idx}  (diverse={best_diversity:.1f}%)')
if best_diversity > 10:
    print(f'→ Use encoder_outputs[{best_stage_idx}] in the re-run cell below.')
else:
    print('→ Even the best stage is collapsed — the Fold 0 _best model has a degenerate encoder.')
    print('  (This would be surprising given the full 170-scan extraction gave cos=0.30.)')
    print('  Try running with more patients or check if val_transforms is deterministic.')


In [ ]:
# ── RE-RUN FULL EXTRACTION with corrected stage index (if needed) ──
#
# Only run this if:
#   (a) Cells 12-14 show a non-collapsed stage OTHER than stage 4 (enc[-1])
#   (b) OR you want to try extracting Folds 1 & 2 with a shallower stage
#
# NOTE: Based on current evidence, Folds 1 & 2 have a TRAINING COLLAPSE.
# The encoder outputs a constant vector at ALL stages → no stage fix helps.
# This cell is kept for completeness / future use.

STAGE_TO_USE = -1  # Change this if Cell 14 found a better stage
FOLDS_TO_RERUN = [1, 2]  # Folds to re-extract


def extract_with_stage(fold, ckpt_type, stage_idx):
    """Full 170-scan extraction using a specific swinViT stage."""
    ckpt_path = CKPT_DIR / f'bsf_fold{fold}_{ckpt_type}.pth'
    out_path  = EMB_DIR  / f'bsf_fold{fold}_{ckpt_type}_stage{stage_idx}.npz'

    if not ckpt_path.exists():
        print(f'  MISSING: {ckpt_path.name}'); return None
    if out_path.exists(): out_path.unlink()

    print(f'\n>>> FOLD {fold} | {ckpt_type} | stage={stage_idx}')
    model = create_model()
    ckpt  = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f'    epoch={ckpt.get("epoch","?")}  dice={ckpt.get("best_dice",float("nan")):.4f}')
    model.load_state_dict(ckpt['model_state_dict'], strict=True)
    model = model.to(device).eval()

    embeddings = {}
    with torch.no_grad():
        for s in tqdm(all_dicts, desc=f'Fold{fold} [{ckpt_type}] stg{stage_idx}'):
            key = f'{s["patient_dir"]}__{s["visit"]}'
            try:
                out  = val_transforms(s)
                imgs = out['image'].unsqueeze(0).float()
                imgs = torch.nn.functional.interpolate(
                    imgs, size=(96,96,96), mode='trilinear', align_corners=False
                ).to(device)
                if torch.cuda.is_available():
                    with torch.amp.autocast('cuda'):
                        enc = model.swinViT(imgs, model.normalize)
                else:
                    enc = model.swinViT(imgs, model.normalize)
                emb = torch.nn.functional.adaptive_avg_pool3d(
                    enc[stage_idx].float(), 1
                ).flatten()
                embeddings[key] = emb.cpu().numpy()
            except Exception as e:
                print(f'    ERR {key}: {e}')

    X     = np.array(list(embeddings.values()))
    norms = np.linalg.norm(X, axis=1)          # row-wise L2 ✅ FIXED
    Xn    = X / (norms[:, None] + 1e-8)
    cos   = Xn @ Xn.T
    mask  = ~np.eye(len(X), dtype=bool)
    cm    = cos[mask].mean()
    dp    = (cos[mask] < 0.95).mean() * 100
    ns    = norms.std()
    status = '✅ VALID' if dp > 10 else '❌ COLLAPSED'
    print(f'    {len(embeddings)} embeddings  |  norm_std={ns:.4f}  cos={cm:.4f}  diverse={dp:.1f}%  {status}')

    np.savez(out_path, **embeddings)
    meta = {k: {'patient_dir': k.split('__')[0], 'visit': k.split('__')[1]} for k in embeddings}
    with open(EMB_DIR / f'bsf_fold{fold}_{ckpt_type}_stage{stage_idx}_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'    Saved: {out_path.name}')

    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return cm, dp, ns


print('='*60)
print(f'RE-RUN with stage={STAGE_TO_USE} for Folds {FOLDS_TO_RERUN}')
print('='*60)
print('NOTE: If Folds 1&2 have training collapse,')
print('      no stage change will fix them — all stages collapse together.')
print()

rerun_results = {}
for fold in FOLDS_TO_RERUN:
    r = extract_with_stage(fold, 'best', STAGE_TO_USE)
    rerun_results[fold] = r

print()
for fold, r in rerun_results.items():
    if r:
        cm, dp, ns = r
        st = '✅ VALID' if dp > 10 else '❌ COLLAPSED'
        print(f'  Fold {fold}: norm_std={ns:.4f}  cos={cm:.4f}  diverse={dp:.1f}%  {st}')


In [ ]:
# ── FINAL CONCLUSION & DIAGNOSIS ──
#
# Based on ALL evidence collected in this notebook:

print('='*65)
print('CONFIRMED DIAGNOSIS: TRAINING MODE COLLAPSE in Folds 1 & 2')
print('='*65)
print()
print('EVIDENCE SUMMARY:')
print('  Fold 0 _best  → 170 scans → cos=0.302  diverse=42.1%  norm_std=5.70  ✅ VALID')
print('  Fold 1 _best  → 170 scans → cos=1.000  diverse=0.0%   norm_std=0.001 ❌ COLLAPSED')
print('  Fold 1 _latest→ 170 scans → cos=0.999  diverse=0.0%   norm_std=0.024 ❌ COLLAPSED')
print('  Fold 2 _best  → 170 scans → cos=1.000  diverse=0.0%   norm_std=0.131 ❌ COLLAPSED')
print('  Fold 2 _latest→ 170 scans → cos=1.000  diverse=0.0%   norm_std=0.131 ❌ COLLAPSED')
print()
print('KEY FACTS:')
print('  1. Extraction code is IDENTICAL to Phase3_A1B (confirmed by code inspection).')
print('  2. Fold 0 _best epoch=59, same as Fold 1/2 → epoch is not the cause.')
print('  3. Norm_std < 0.001 for Fold 1 = encoder outputs LITERALLY IDENTICAL vectors.')
print('  4. ALL encoder stages (0-4) collapsed for Folds 1 & 2.')
print('  5. The quick-check (20 consecutive scans) was misleading: those are same-patient')
print('     longitudinal scans with naturally high similarity (cos=0.86).')
print()
print('ROOT CAUSE (hypothesis):')
print('  Folds 1 & 2 trained with different patient splits → different data distribution.')
print('  The model found a degenerate minimum where the encoder outputs a constant')
print('  response regardless of input — it can still achieve decent Dice (~0.50)')
print('  by relying on the decoder with a fixed latent code.')
print('  This is a known failure mode in self-supervised / segmentation ViTs with')
print('  small datasets and no explicit representation diversity loss.')
print()
print('WHAT TO DO NEXT:')
print('  SHORT-TERM (thesis deadline):')
print('    • Use Fold 0 embeddings only — they are scientifically valid.')
print('    • Report 1-fold results with honest note about fold instability.')
print('    • Download: bsf_embeddings_fold0_best.npz (already saved, valid)')
print()
print('  MEDIUM-TERM (retrain Folds 1 & 2):')
print('    • Lower learning rate: 1e-5 (from 1e-4)')
print('    • Add learning rate warmup (10 epochs)')
print('    • Add gradient clipping: max_norm=1.0')
print('    • Consider adding a contrastive loss or representation diversity term')
print()
print('FILES TO DOWNLOAD:')
print('  embeddings/bsf_embeddings_fold0_best.npz  ← USE THIS for evaluation')
print()
print('  After download:')
print('  cp bsf_embeddings_fold0_best.npz')
print('     implementation_cyprus/Phase3/bsf_fold_outputs/embeddings/bsf_embeddings_fold0.npz')
print('  (Fold 0 original is already valid — this just confirms it)')


In [12]:
# ── Cell 11: Compare diversity — original (collapsed) vs final epoch ──

print('=== Full Diversity Comparison ===')
print(f'{"Fold":<5} {"Type":<12} {"Cos mean":<12} {"% diverse":<14} {"Norm std":<12} {"Status"}')
print('-' * 70)

# Load original collapsed embeddings from the BSF dataset
ORIG_DIR = None
for input_dir in Path('/kaggle/input').iterdir():
    candidates = list(input_dir.rglob('bsf_embeddings_fold0.npz'))
    if candidates:
        ORIG_DIR = candidates[0].parent
        break

for fold in range(3):
    rows = []
    # Original
    if ORIG_DIR:
        op = ORIG_DIR / f'bsf_embeddings_fold{fold}.npz'
        if op.exists():
            rows.append(('original', op))
    # New latest
    rows.append(('latest', EMB_DIR / f'bsf_embeddings_fold{fold}_latest.npz'))

    for label, fpath in rows:
        if not fpath.exists():
            print(f'{fold:<5} {label:<12} NOT FOUND')
            continue
        d   = np.load(fpath)
        X   = np.array([d[k] for k in sorted(d.keys())])
        norms = np.linalg.norm(X, axis=1)
        Xn  = X / (norms[:, None] + 1e-8)
        cos = Xn @ Xn.T
        mask = ~np.eye(len(X), dtype=bool)
        cm  = cos[mask].mean()
        dp  = (cos[mask] < 0.95).mean() * 100
        st  = '✅ Valid' if dp > 10 else '❌ Collapsed'
        print(f'{fold:<5} {label:<12} {cm:<12.4f} {dp:<14.1f}% {norms.std():<12.4f} {st}')

=== Full Diversity Comparison ===
Fold  Type         Cos mean     % diverse      Norm std     Status
----------------------------------------------------------------------
0     original     0.3023       42.1          % 5.6973       ✅ Valid
0     latest       NOT FOUND
1     original     1.0000       0.0           % 0.0011       ❌ Collapsed
1     latest       0.9999       0.0           % 0.0237       ❌ Collapsed
2     original     1.0000       0.0           % 0.1310       ❌ Collapsed
2     latest       1.0000       0.0           % 0.1310       ❌ Collapsed


In [13]:
# ── Cell 12: Same-patient vs cross-patient separation ──
# The KEY diagnostic: BSF should cluster same-patient scans together (Δ ≈ 0.57)

print('=== Same-Patient vs Cross-Patient Cosine Separation ===')
print(f'{"Fold":<5} {"Type":<12} {"Same-pat":<12} {"Cross-pat":<12} {"Δ":<10} {"BL→lastFU"}')
print('-' * 65)

import json as json_
def get_pid(key): return key.split('__')[0].rstrip('ab')

# Load timelines from dataset
tl_path = None
for input_dir in Path('/kaggle/input').iterdir():
    for p in input_dir.rglob('cyprus_patient_timelines.csv'):
        tl_path = p; break
    if tl_path: break

timelines = None
if tl_path:
    import pandas as pd
    timelines = pd.read_csv(tl_path)

for fold in range(3):
    for label, fpath in [
        ('original', ORIG_DIR / f'bsf_embeddings_fold{fold}.npz' if ORIG_DIR else None),
        ('latest',   EMB_DIR  / f'bsf_embeddings_fold{fold}_latest.npz'),
    ]:
        if fpath is None or not fpath.exists(): continue
        d = np.load(fpath)
        keys = sorted(d.keys())
        X = np.array([d[k] for k in keys])
        norms = np.linalg.norm(X, axis=1)
        Xn = X / (norms[:, None] + 1e-8)
        cos = Xn @ Xn.T

        same, cross = [], []
        for i in range(len(keys)):
            for j in range(i+1, len(keys)):
                if get_pid(keys[i]) == get_pid(keys[j]):
                    same.append(cos[i,j])
                else:
                    cross.append(cos[i,j])

        # Baseline → last FU
        bl_fu = []
        if timelines is not None:
            keys_set = set(keys)
            for _,grp in timelines.groupby('patient_id'):
                grp = grp.sort_values('visit_idx')
                if len(grp) < 2: continue
                pid = grp.iloc[0]['patient_id']
                k1 = f'{pid}__{grp.iloc[0]["visit_name"]}'
                k2 = f'{pid}__{grp.iloc[-1]["visit_name"]}'
                if k1 in keys_set and k2 in keys_set:
                    i1, i2 = keys.index(k1), keys.index(k2)
                    bl_fu.append(cos[i1, i2])

        bl_str = f'{np.mean(bl_fu):.4f}' if bl_fu else 'N/A'
        delta  = np.mean(same) - np.mean(cross)
        print(f'{fold:<5} {label:<12} {np.mean(same):<12.4f} {np.mean(cross):<12.4f} {delta:<10.4f} {bl_str}')

print('\n  Fold 0 reference: same=0.862, cross=0.288, Δ=0.574, BL→lastFU=0.755')

=== Same-Patient vs Cross-Patient Cosine Separation ===
Fold  Type         Same-pat     Cross-pat    Δ          BL→lastFU
-----------------------------------------------------------------
0     original     0.8620       0.2876       0.5744     0.9999
1     original     1.0000       1.0000       0.0000     1.0000
1     latest       1.0000       0.9999       0.0001     1.0000
2     original     1.0000       1.0000       0.0000     1.0000
2     latest       1.0000       1.0000       0.0000     1.0000

  Fold 0 reference: same=0.862, cross=0.288, Δ=0.574, BL→lastFU=0.755


In [14]:
# ── Cell 13: List output files for download ──
print('=== Output Files (download from Kaggle output tab) ===')
total_bytes = 0
for p in sorted(OUTPUT_ROOT.rglob('*')):
    if p.is_file():
        mb = p.stat().st_size / 1024 / 1024
        total_bytes += p.stat().st_size
        print(f'  {p.relative_to(OUTPUT_ROOT)}  ({mb:.2f} MB)')
print(f'\nTotal: {total_bytes/1024/1024:.1f} MB')

print()
print('=== After downloading, replace local files with: ===')
print('cp bsf_embeddings_fold1_latest.npz \\')
print('   implementation_cyprus/Phase3/bsf_fold_outputs/embeddings/bsf_embeddings_fold1.npz')
print('cp bsf_embeddings_fold2_latest.npz \\')
print('   implementation_cyprus/Phase3/bsf_fold_outputs/embeddings/bsf_embeddings_fold2.npz')
print()
print('Then re-run Phase3_A4_Embedding_Eval.ipynb for updated 16-test scores.')

=== Output Files (download from Kaggle output tab) ===
  checkpoints/bsf_fold0_best.pth  (719.10 MB)
  checkpoints/bsf_fold0_latest.pth  (719.11 MB)
  checkpoints/bsf_fold1_best.pth  (719.10 MB)
  checkpoints/bsf_fold1_latest.pth  (719.11 MB)
  checkpoints/bsf_fold2_best.pth  (719.10 MB)
  checkpoints/bsf_fold2_latest.pth  (719.11 MB)
  embeddings/bsf_embeddings_fold1_latest.npz  (0.54 MB)
  embeddings/bsf_embeddings_fold1_latest_meta.json  (0.01 MB)
  embeddings/bsf_embeddings_fold2_latest.npz  (0.54 MB)
  embeddings/bsf_embeddings_fold2_latest_meta.json  (0.01 MB)

Total: 4315.7 MB

=== After downloading, replace local files with: ===
cp bsf_embeddings_fold1_latest.npz \
   implementation_cyprus/Phase3/bsf_fold_outputs/embeddings/bsf_embeddings_fold1.npz
cp bsf_embeddings_fold2_latest.npz \
   implementation_cyprus/Phase3/bsf_fold_outputs/embeddings/bsf_embeddings_fold2.npz

Then re-run Phase3_A4_Embedding_Eval.ipynb for updated 16-test scores.


## Kaggle Run Instructions

### Datasets required
1. `mohamedmohamed23/brainsegfounder-fold-outputs` — contains all `_best.pth` and `_latest.pth` checkpoints
2. `mohamedmohamed23/cyprus-proteas-brain-mets` — MRI scans + `data_splits.json`

### Step-by-step

1. **Run Cells 1–10** (top to bottom) — setup, packages, data loading, model definition
2. **Run Cell 11** (MAIN) — already done; Folds 1&2 from `_latest` are collapsed
3. **Run Cell 12** (DIAGNOSTIC) — quick check whether `_best.pth` gives diverse embeddings
4. **If Cell 12 shows `_best` is valid → Run Cell 13** (FULL EXTRACTION from `_best`)
5. **If Cell 12 shows everything collapsed → Run Cells 14 → 15** (deep encoder inspection)
6. **Run Cells 16–18** — comparison tables + list output files
7. **Download outputs** from Kaggle Output tab

### After downloading

```bash
# If _best embeddings are valid:
cp bsf_embeddings_fold1_best.npz \
   implementation_cyprus/Phase3/bsf_fold_outputs/embeddings/bsf_embeddings_fold1.npz
cp bsf_embeddings_fold2_best.npz \
   implementation_cyprus/Phase3/bsf_fold_outputs/embeddings/bsf_embeddings_fold2.npz

# Then re-run evaluation:
# Phase3_A4_Embedding_Eval.ipynb
```

### Troubleshooting

| Problem | Fix |
|---------|-----|
| `bsf_fold*_best.pth` not found | Add `mohamedmohamed23/brainsegfounder-fold-outputs` dataset |
| 0 scans resolved | Check Cell 5 debug output — file structure vs `data_splits.json` paths |
| All embeddings collapsed (even `_best`) | Run Cell 14 — check which encoder stage is diverse |
| `SwinUNETR` `img_size` error | Should be auto-handled by `_has_img_size` check |
| OOM on T4 | `cache_rate=0.0` is already set; try CPU if needed |

### What the results mean

| Metric | Good | Collapsed |
|--------|------|-----------|
| Pairwise cosine mean | < 0.5 | > 0.99 |
| % diverse pairs (cos < 0.95) | > 10% | < 1% |
| Norm std | > 1.0 | < 0.1 |
| Same-pat vs cross-pat Δ | > 0.3 | ≈ 0.0 |
